# Wadjet v2 — Hieroglyph Classifier Training (ONNX Export)

**Purpose**: Train a MobileNetV3-Small classifier for 171 Gardiner hieroglyph signs, export as ONNX for browser + server.

## CRITICAL RULES
1. **float32 ONLY** — NO `mixed_float16` (causes fused ops that crash in any runtime)
2. Export via **`tf2onnx.convert.from_keras()`** → `.onnx` file
3. Quantize via **`onnxruntime.quantization.quantize_dynamic()`** → uint8
4. Load in browser with **`ort.InferenceSession.create()`** (ONNX Runtime Web)
5. **NO horizontal flip** augmentation (hieroglyphs are direction-sensitive)
6. Architecture: **MobileNetV3-Small** at **128×128**

## Dataset
- Source: `naderelakany/wadjet-tfrecords` on Kaggle
- 171 classes (Gardiner signs), 15,017 train / 1,152 val / 1,154 test images
- TFRecords at 384×384 JPEG, resized to 128×128 during training

## Training Strategy
- **Phase 1**: Head only (5 epochs, backbone frozen, lr=1e-3)
- **Phase 2**: Fine-tune top 70% (30 epochs, lr=1e-4 cosine decay)
- Focal Loss (γ=2.0) + sqrt-inverse-frequency class weights

In [ ]:
# ============================================================
# Auto-Detect Dataset Path
# ============================================================
import os, glob

DATASET_SLUG = "wadjet-hieroglyph-tfrecords"

def find_dataset(slug):
    """Search /kaggle/input for the dataset, handling nested paths."""
    candidates = [
        f'/kaggle/input/{slug}',                           # Standard
        f'/kaggle/input/datasets/naderelakany/{slug}',     # Cross-user
    ]
    # Also search recursively
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == slug:
            candidates.append(root)
    
    for path in candidates:
        if os.path.isdir(path):
            contents = os.listdir(path)
            print(f'  Found: {path} ({len(contents)} items)')
            for f in sorted(contents)[:5]:
                print(f'    {f}')
            return path
    
    # Last resort: list everything under /kaggle/input
    print('ERROR: Dataset not found. Contents of /kaggle/input:')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        for f in files[:5]:
            print(f'{indent}  {f}')
    raise FileNotFoundError(f'Dataset {slug} not found in /kaggle/input')

DATA_DIR = find_dataset(DATASET_SLUG)
print(f'\nDATA_DIR = {DATA_DIR}')


In [ ]:
# ============================================================
# Configuration
# ============================================================
import os

# Data paths — files are flat: train_*.tfrecord etc.
# DATA_DIR set by auto-detect cell above
TRAIN_DIR = DATA_DIR  # flat layout: train_*.tfrecord in root
VAL_DIR = DATA_DIR    # flat layout: val_*.tfrecord in root
TEST_DIR = DATA_DIR   # flat layout: test_*.tfrecord in root

# Model
IMAGE_SIZE = 128
NUM_CLASSES = 171

# Training
BATCH_SIZE = 32
SEED = 42
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 30
HEAD_LR = 1e-3
FINETUNE_LR = 1e-4
FINETUNE_MIN_LR = 5e-6
WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
FOCAL_GAMMA = 2.0
UNFREEZE_RATIO = 0.7
PATIENCE = 15

# Output
OUTPUT_DIR = "/kaggle/working"
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model.keras")
ONNX_FLOAT_PATH = os.path.join(OUTPUT_DIR, "hieroglyph_classifier.onnx")
ONNX_UINT8_PATH = os.path.join(OUTPUT_DIR, "hieroglyph_classifier_uint8.onnx")
METADATA_PATH = os.path.join(OUTPUT_DIR, "model_metadata.json")

# WandB (set True if configured)
USE_WANDB = False
WANDB_PROJECT = "wadjet-hieroglyph-v2"

In [ ]:
# ============================================================
# Import (no pip install — use pre-installed packages only)
# ============================================================
import glob
import json
import math
import os
import numpy as np
import tensorflow as tf
import keras
from pathlib import Path

tf.random.set_seed(SEED)
np.random.seed(SEED)

# Check if tf2onnx is available (for ONNX export at end)
try:
    import tf2onnx
    HAS_TF2ONNX = True
    print(f'tf2onnx available: {tf2onnx.__version__}')
except ImportError:
    HAS_TF2ONNX = False
    print('tf2onnx NOT available — will save SavedModel for local ONNX conversion')

try:
    import onnxruntime
    HAS_ORT = True
    print(f'onnxruntime available: {onnxruntime.__version__}')
except ImportError:
    HAS_ORT = False
    print('onnxruntime NOT available — will skip ONNX validation')


In [ ]:
# ============================================================
# GPU + float32 Verification
# ============================================================
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {tf.keras.__version__}")

# CRITICAL: Verify NO mixed precision
policy = tf.keras.mixed_precision.global_policy()
print(f"Global dtype policy: {policy}")
assert 'float16' not in str(policy), "ABORT: mixed_float16 detected! Must use float32."

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")
for gpu in gpus:
    print(f"  {gpu}")

In [ ]:
# ============================================================
# TFRecord Parsing
# ============================================================
FEATURE_DESC = {
    "image_raw": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.int64),
}

def parse_example(example_proto):
    """Parse a single TFRecord example."""
    features = tf.io.parse_single_example(example_proto, FEATURE_DESC)
    image = tf.io.decode_jpeg(features["image_raw"], channels=3)
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    label = tf.cast(features["label"], tf.int32)
    return image, label

def load_dataset(split_dir, batch_size=BATCH_SIZE, augment=False, shuffle=False, split_prefix=None):
    """Load TFRecords from a directory. If split_prefix given, glob split_prefix_*.tfrecord."""
    pattern = f"{split_prefix}_*.tfrecord" if split_prefix else "*.tfrecord"
    files = sorted(glob.glob(os.path.join(split_dir, pattern)))
    assert len(files) > 0, f"No TFRecord files found in {split_dir}"
    print(f"  Loading {len(files)} TFRecord files from {split_dir}")

    ds = tf.data.TFRecordDataset(files, num_parallel_reads=tf.data.AUTOTUNE)
    ds = ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(1024, seed=SEED)
    if augment:
        ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

In [ ]:
# ============================================================
# Dataset Statistics
# ============================================================
print("Counting samples per split...")
train_count = sum(1 for _ in tf.data.TFRecordDataset(
    sorted(glob.glob(os.path.join(TRAIN_DIR, "train_*.tfrecord")))))
val_count = sum(1 for _ in tf.data.TFRecordDataset(
    sorted(glob.glob(os.path.join(VAL_DIR, "val_*.tfrecord")))))
test_count = sum(1 for _ in tf.data.TFRecordDataset(
    sorted(glob.glob(os.path.join(TEST_DIR, "test_*.tfrecord")))))

print(f"Train: {train_count:,} | Val: {val_count:,} | Test: {test_count:,}")
print(f"Total: {train_count + val_count + test_count:,}")

In [ ]:
# ============================================================
# Data Augmentation (NO horizontal flip)
# ============================================================
def rotate_image(image, angle):
    """Rotate image by angle (radians) using projective transform."""
    cos_a = tf.cos(angle)
    sin_a = tf.sin(angle)
    h = tf.cast(tf.shape(image)[0], tf.float32)
    w = tf.cast(tf.shape(image)[1], tf.float32)
    cx, cy = w / 2.0, h / 2.0
    transform = [
        cos_a, -sin_a, cx - cx * cos_a + cy * sin_a,
        sin_a,  cos_a, cy - cx * sin_a - cy * cos_a,
        0.0, 0.0
    ]
    transformed = tf.raw_ops.ImageProjectiveTransformV3(
        images=tf.expand_dims(image, 0),
        transforms=tf.reshape(transform, [1, 8]),
        output_shape=tf.shape(image)[:2],
        interpolation="BILINEAR",
        fill_mode="REFLECT",
        fill_value=0.0
    )
    return transformed[0]

def augment_image(image, label):
    """Online augmentation — NO horizontal flip (hieroglyphs are direction-sensitive)."""
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.image.random_saturation(image, 0.8, 1.2)
    # Rotation: +/-10 degrees
    angle = tf.random.uniform([], -10.0, 10.0) * (math.pi / 180.0)
    image = rotate_image(image, angle)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

In [ ]:
# ============================================================
# Build Datasets
# ============================================================
print("Building datasets...")
train_ds = load_dataset(TRAIN_DIR, augment=True, shuffle=True, split_prefix="train")
val_ds = load_dataset(VAL_DIR, augment=False, shuffle=False, split_prefix="val")
test_ds = load_dataset(TEST_DIR, augment=False, shuffle=False, split_prefix="test")

# Quick sanity check
for images, labels in train_ds.take(1):
    print(f"Batch shape: {images.shape}, dtype: {images.dtype}")
    print(f"Labels shape: {labels.shape}, range: [{labels.numpy().min()}, {labels.numpy().max()}]")
    assert images.dtype == tf.float32, "Images must be float32!"
    print("Sanity check passed.")

In [ ]:
# ============================================================
# Focal Loss
# ============================================================
import keras

@keras.saving.register_keras_serializable()
class FocalLoss(tf.keras.losses.Loss):
    """Sparse categorical focal loss (integer labels).
    Registered for Keras serialization — required for:
    - model.save() to .keras format
    - tf2onnx.convert.from_keras() (ONNX export)
    """
    def __init__(self, gamma=2.0, name="focal_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = gamma

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        batch_indices = tf.range(tf.shape(y_true)[0])
        indices = tf.stack([batch_indices, y_true], axis=1)
        p_t = tf.gather_nd(y_pred, indices)
        focal_weight = tf.pow(1.0 - p_t, self.gamma)
        ce = -tf.math.log(p_t)
        return tf.reduce_mean(focal_weight * ce)

    def get_config(self):
        config = super().get_config()
        config.update({"gamma": self.gamma})
        return config

In [ ]:
# ============================================================
# Class Weights (sqrt-inverse-frequency)
# ============================================================
print("Computing class weights...")
counts = np.zeros(NUM_CLASSES, dtype=np.float64)
for _, labels in load_dataset(TRAIN_DIR, batch_size=512, augment=False, shuffle=False, split_prefix="train"):
    for lbl in labels.numpy():
        counts[lbl] += 1

total = counts.sum()
weights = np.where(counts > 0, np.sqrt(total / (NUM_CLASSES * counts)), 1.0)
weights = weights / weights.mean()  # normalize to mean=1
CLASS_WEIGHTS = {i: float(w) for i, w in enumerate(weights)}

print(f"Total samples: {int(total)}")
print(f"Weight range: [{min(CLASS_WEIGHTS.values()):.2f}, {max(CLASS_WEIGHTS.values()):.2f}]")
print(f"Classes with highest weight (rare):")
sorted_weights = sorted(CLASS_WEIGHTS.items(), key=lambda x: -x[1])[:10]
for idx, w in sorted_weights:
    print(f"  Class {idx}: weight={w:.2f}, count={int(counts[idx])}")

In [ ]:
# ============================================================
# Model Architecture: MobileNetV3-Small
# ============================================================
def build_model(num_classes, image_size, dropout=DROPOUT, freeze_base=True):
    """Build MobileNetV3-Small with custom classification head."""
    base = tf.keras.applications.MobileNetV3Small(
        input_shape=(image_size, image_size, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = not freeze_base

    inputs = tf.keras.Input(shape=(image_size, image_size, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    # Always float32 output for numerical stability
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = tf.keras.Model(inputs, outputs, name="wadjet_hieroglyph_classifier")
    return model, base

model, backbone = build_model(NUM_CLASSES, IMAGE_SIZE, freeze_base=True)
model.summary(show_trainable=True, expand_nested=False)

In [ ]:
# ============================================================
# Phase 1: Head Training (backbone frozen)
# ============================================================
print("=" * 60)
print("PHASE 1: Training classification head (backbone frozen)")
print("=" * 60)

model.compile(
    optimizer=tf.keras.optimizers.Adam(HEAD_LR),
    loss=FocalLoss(gamma=FOCAL_GAMMA),
    metrics=["accuracy"],
)

callbacks_p1 = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1
    ),
]
if USE_WANDB:
    import wandb
    from wandb.integration.keras import WandbMetricsLogger
    wandb.init(project=WANDB_PROJECT, name="phase1-head", config={
        "image_size": IMAGE_SIZE, "backbone": "MobileNetV3Small",
        "phase": 1, "lr": HEAD_LR, "epochs": HEAD_EPOCHS,
    })
    callbacks_p1.append(WandbMetricsLogger(log_freq="epoch"))

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_p1,
)

print(f"Phase 1 complete. Best val_accuracy: {max(history_p1.history['val_accuracy']):.4f}")
if USE_WANDB:
    wandb.finish()

In [ ]:
# ============================================================
# Phase 2: Fine-tuning (top 70% of backbone unfrozen)
# ============================================================
print("=" * 60)
print("PHASE 2: Fine-tuning (unfreezing top 70% of backbone)")
print("=" * 60)

# Unfreeze top 70% of backbone
backbone.trainable = True
freeze_until = int(len(backbone.layers) * (1.0 - UNFREEZE_RATIO))
for layer in backbone.layers[:freeze_until]:
    layer.trainable = False

trainable_count = sum(1 for l in backbone.layers if l.trainable)
total_layers = len(backbone.layers)
print(f"Backbone: {trainable_count}/{total_layers} layers trainable")

# Cosine decay LR schedule
steps_per_epoch = train_count // BATCH_SIZE + 1
total_steps = steps_per_epoch * FINETUNE_EPOCHS
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=FINETUNE_LR,
    decay_steps=total_steps,
    alpha=FINETUNE_MIN_LR / FINETUNE_LR,
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=lr_schedule, weight_decay=WEIGHT_DECAY
    ),
    loss=FocalLoss(gamma=FOCAL_GAMMA),
    metrics=["accuracy"],
)

callbacks_p2 = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE,
        min_delta=0.001, restore_best_weights=True, verbose=1
    ),
]
if USE_WANDB:
    import wandb
    from wandb.integration.keras import WandbMetricsLogger
    wandb.init(project=WANDB_PROJECT, name="phase2-finetune", config={
        "image_size": IMAGE_SIZE, "backbone": "MobileNetV3Small",
        "phase": 2, "lr": FINETUNE_LR, "epochs": FINETUNE_EPOCHS,
        "unfreeze_ratio": UNFREEZE_RATIO
    })
    callbacks_p2.append(WandbMetricsLogger(log_freq="epoch"))

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_p2,
)

print(f"Phase 2 complete. Best val_accuracy: {max(history_p2.history['val_accuracy']):.4f}")
if USE_WANDB:
    wandb.finish()

In [ ]:
# ============================================================
# Evaluation + Per-Class Metrics
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix

# Load best model
model = tf.keras.models.load_model(
    BEST_MODEL_PATH, custom_objects={"FocalLoss": FocalLoss}
)

# Predict on test set
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Overall accuracy
accuracy = np.mean(y_true == y_pred)
print(f"Test accuracy: {accuracy:.4f}")

# Per-class F1 (top 20 worst classes)
report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
class_f1 = {int(k): v['f1-score'] for k, v in report.items() if k.isdigit()}
worst_classes = sorted(class_f1.items(), key=lambda x: x[1])[:20]
print("\nWorst 20 classes by F1-score:")
for idx, f1 in worst_classes:
    print(f"  Class {idx}: F1={f1:.3f}, count={int(counts[idx])}")

In [ ]:
# ============================================================
# ONNX Export (or SavedModel fallback)
# ============================================================
SAVEDMODEL_PATH = os.path.join(OUTPUT_DIR, 'hieroglyph_classifier_savedmodel')

if HAS_TF2ONNX:
    print('Exporting to ONNX via tf2onnx...')
    spec = (tf.TensorSpec((None, IMAGE_SIZE, IMAGE_SIZE, 3), tf.float32, name='input'),)
    model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=13)
    with open(ONNX_FLOAT_PATH, 'wb') as f:
        f.write(model_proto.SerializeToString())
    print(f'  Saved ONNX float32: {ONNX_FLOAT_PATH}')
    print(f'  Size: {os.path.getsize(ONNX_FLOAT_PATH) / 1024 / 1024:.1f} MB')
else:
    print('tf2onnx not available. Saving as SavedModel for local conversion...')
    tf.saved_model.save(model, SAVEDMODEL_PATH)
    print(f'  Saved: {SAVEDMODEL_PATH}')
    # Also save as .keras
    model.save(BEST_MODEL_PATH)
    print(f'  Saved .keras: {BEST_MODEL_PATH}')


In [ ]:
# ============================================================
# ONNX uint8 Quantization (skip if unavailable)
# ============================================================
if HAS_TF2ONNX and HAS_ORT:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(ONNX_FLOAT_PATH, ONNX_UINT8_PATH, weight_type=QuantType.QUInt8)
    print(f'  Quantized ONNX: {ONNX_UINT8_PATH}')
    print(f'  Size: {os.path.getsize(ONNX_UINT8_PATH) / 1024 / 1024:.1f} MB')
else:
    print('Skipping ONNX quantization (tf2onnx/onnxruntime not available)')
    print('Will do ONNX conversion + quantization locally after download.')


In [ ]:
# ============================================================
# Validate ONNX Model (skip if unavailable)
# ============================================================
if HAS_ORT and HAS_TF2ONNX:
    import onnxruntime as ort
    session = ort.InferenceSession(ONNX_UINT8_PATH, providers=['CPUExecutionProvider'])
    input_meta = session.get_inputs()[0]
    output_meta = session.get_outputs()[0]
    print(f'Input:  {input_meta.name} shape={input_meta.shape} dtype={input_meta.type}')
    print(f'Output: {output_meta.name} shape={output_meta.shape} dtype={output_meta.type}')
    
    dummy = np.random.rand(1, IMAGE_SIZE, IMAGE_SIZE, 3).astype(np.float32)
    out = session.run(None, {input_meta.name: dummy})
    print(f'Test output shape: {out[0].shape}, sum: {out[0].sum():.4f}')
    print('✅ ONNX model validated successfully')
else:
    print('Skipping ONNX validation (packages not available)')


In [ ]:
# ============================================================
# Save Model Metadata + All Outputs
# ============================================================
# Load label mapping from the Kaggle dataset
label_map_path = os.path.join(DATA_DIR, "classification", "metadata.json")
if not os.path.exists(label_map_path):
    # Try alternate location
    label_map_path = os.path.join(DATA_DIR, "metadata.json")

if os.path.exists(label_map_path):
    with open(label_map_path) as f:
        metadata = json.load(f)
    print(f"Loaded metadata from {label_map_path}")
else:
    metadata = {"num_classes": NUM_CLASSES, "image_size": IMAGE_SIZE}
    print("No metadata.json found, using defaults")

# Determine what format was actually exported
has_onnx = os.path.exists(ONNX_FLOAT_PATH)
has_onnx_q = os.path.exists(ONNX_UINT8_PATH)
export_format = "onnx" if has_onnx else "savedmodel"

# Create output metadata
output_metadata = {
    "model_name": "wadjet_hieroglyph_classifier_v2",
    "architecture": "MobileNetV3Small",
    "input_size": IMAGE_SIZE,
    "num_classes": NUM_CLASSES,
    "format": export_format,
    "load_method": "ort.InferenceSession.create(url)" if has_onnx else "convert SavedModel to ONNX locally",
    "preprocessing": {
        "normalize": "divide_by_255",
        "input_range": [0.0, 1.0],
    },
    "training": {
        "precision": "float32",
        "loss": f"FocalLoss(gamma={FOCAL_GAMMA})",
        "head_epochs": HEAD_EPOCHS,
        "finetune_epochs": FINETUNE_EPOCHS,
        "test_accuracy": float(accuracy),
    },
    "original_metadata": metadata,
}

with open(METADATA_PATH, "w") as f:
    json.dump(output_metadata, f, indent=2)
print(f"Metadata saved to: {METADATA_PATH}")

# Save .keras model (backup)
model.save(BEST_MODEL_PATH)
print(f"Keras model saved to: {BEST_MODEL_PATH}")

# Summary of all outputs
print("\n" + "=" * 60)
print("ALL OUTPUTS:")
print("=" * 60)
print(f"  1. Keras model (backup): {BEST_MODEL_PATH}")
if has_onnx:
    print(f"  2. ONNX float32: {ONNX_FLOAT_PATH} ({os.path.getsize(ONNX_FLOAT_PATH)/(1024*1024):.1f} MB)")
else:
    sm_path = os.path.join(OUTPUT_DIR, 'hieroglyph_classifier_savedmodel')
    print(f"  2. SavedModel: {sm_path}")
if has_onnx_q:
    print(f"  3. ONNX uint8: {ONNX_UINT8_PATH} ({os.path.getsize(ONNX_UINT8_PATH)/(1024*1024):.1f} MB)")
else:
    print(f"  3. ONNX uint8: (not exported — convert locally)")
print(f"  4. Metadata: {METADATA_PATH}")


## Download Instructions

After training completes on Kaggle:

1. **Download `hieroglyph_classifier_uint8.onnx`** (recommended for production — single file!)

2. **Download `model_metadata.json`**

3. **Copy to Wadjet v2 project**:
   ```
   models/hieroglyph/classifier/
   ├── hieroglyph_classifier_uint8.onnx
   └── model_metadata.json
   ```

4. **Update `hieroglyph-pipeline.js`**:
   - Replace `tf.loadGraphModel()` with `ort.InferenceSession.create('/models/hieroglyph/classifier/hieroglyph_classifier_uint8.onnx')`
   - Update preprocessing: resize to 128×128 → /255.0 → NHWC format → `new ort.Tensor('float32', data, [1,128,128,3])`

5. **Remove TF.js CDN** from `scan.html` (ONNX Runtime Web is already loaded for detector)

6. **Bump service worker cache version**

7. **Test in browser**: Model should load via `ort.InferenceSession.create()` — no more `_FusedConv2D` errors!